# CLI Plot Walkthrough

**Model.** This notebook demonstrates the `quantum-lattice` command-line interface. The CLI builds package models, prints spectra, and saves spectrum plots.

**Typical uses.** Quick smoke tests, reproducible command-line figures, simple generated artifacts for docs, and examples that do not require writing a Python script.

**Parameters.** `--model` selects a registered builder. Common options include `--n-sites`, `--n-cells`, `--n-rows`, `--n-cols`, `--hopping`, `--flux`, and `--output`.

**Useful plots.** CLI-generated PNG spectra saved under `results/notebooks/` or `images/`.

In [1]:
import subprocess
import sys
from pathlib import Path

repository_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
output_dir = repository_root / "results/notebooks"
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
models = subprocess.run(
    [sys.executable, "-m", "quantum_lattice_models.cli", "models"],
    check=True,
    capture_output=True,
    text=True,
)
registry_rows = [line.split("\t", maxsplit=3) for line in models.stdout.splitlines()[:8]]
print("First eight registered models")
print("model                                      | category      | dimension")
print("------------------------------------------ | ------------- | -------------------------")
for name, category, dimension, _description in registry_rows:
    print(f"{name:<42s} | {category:<13s} | {dimension}")

First eight registered models
model                                      | category      | dimension
------------------------------------------ | ------------- | -------------------------
aubry_andre_harper_chain                   | tight_binding | n_sites
bose_hubbard_chain                         | hubbard       | (max_occupancy+1)**n_sites
bose_hubbard_chain_sparse                  | hubbard       | (max_occupancy+1)**n_sites
custom_tight_binding                       | user          | n_sites
custom_tight_binding_sparse                | user          | n_sites
fermi_hubbard_chain                        | hubbard       | 2**(2*n_sites)
fermi_hubbard_chain_sparse                 | hubbard       | 2**(2*n_sites)
haldane_honeycomb_lattice                  | topological   | 2*n_rows*n_cols


In [3]:
spectrum = subprocess.run(
    [
        sys.executable,
        "-m",
        "quantum_lattice_models.cli",
        "spectrum",
        "--model",
        "ssh_model",
        "--n-cells",
        "4",
    ],
    check=True,
    capture_output=True,
    text=True,
)
energies = [float(line) for line in spectrum.stdout.splitlines() if line.strip()]
print("SSH spectrum from CLI")
print("index | energy")
print("--- | ---")
for index, energy in enumerate(energies):
    print(f"{index:>5d} | {energy: .6f}")

SSH spectrum from CLI
index | energy
--- | ---
    0 | -1.413419
    1 | -1.166123
    2 | -0.800098
    3 | -0.047394
    4 |  0.047394
    5 |  0.800098
    6 |  1.166123
    7 |  1.413419


In [4]:
plot_path = output_dir / "cli_hofstadter_spectrum.png"
plot = subprocess.run(
    [
        sys.executable,
        "-m",
        "quantum_lattice_models.cli",
        "plot",
        "--model",
        "harper_hofstadter_square_lattice",
        "--n-rows",
        "4",
        "--n-cols",
        "4",
        "--flux",
        "0.25",
        "--output",
        str(plot_path),
    ],
    check=True,
    capture_output=True,
    text=True,
)
print("CLI plot artifact")
print(f"  file:   {plot_path.relative_to(repository_root)}")
print(f"  exists: {plot_path.exists()}")
print(f"  size:   {plot_path.stat().st_size / 1024:.1f} KiB")

CLI plot artifact
  file:   results/notebooks/cli_hofstadter_spectrum.png
  exists: True
  size:   24.9 KiB
